## Merge line data with scatter data to sort them into categories

Datapoints get :

- level: n80, n90, n70, n10, n20, n30, -

- olulisus: p-value

- annotated word count (form, ekilex_tag=location)

- not annotated word count (form, ekilex_tag is null)

- unique lemma count (ekilex_tag=location)


Updated scatter_lines_df3 can be used to later extract info for gpt

In [1]:
import sqlite3
import pandas as pd
from tqdm import tqdm
import matplotlib.pyplot as plt
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from scipy.stats import binom
import copy

## andmetabelid

In [2]:
filename = "../drive_data/v33_koondkorpus_transaktsioonid.db"
conn = sqlite3.connect(filename)
cursor = conn.cursor()

In [3]:
# graafiku joonte info
query = f"SELECT * FROM scatter_lines_df3"

lines_df = pd.read_sql(query, conn)


In [4]:
lines_df

,x,y_pos80,log2_x,y_neg80,log2_y_pos80,y_pos90,y_neg90,log2_y_pos90,y_pos70,y_neg70,log2_y_pos70,y_pos30,y_neg30,log2_y_pos30,y_pos20,y_neg20,log2_y_pos20,y_pos10,y_neg10,log2_y_pos10
0,1,1,0.000000,0,inf,1,0,inf,0,1,-inf,1,0,inf,0,1,-inf,0,1,-inf
1,2,2,1.000000,0,inf,2,0,inf,0,2,-inf,2,0,inf,0,2,-inf,0,2,-inf
2,3,3,1.584963,0,inf,3,0,inf,1,2,-1.000000,3,0,inf,0,3,-inf,0,3,-inf
3,4,4,2.000000,0,inf,4,0,inf,1,3,-1.584963,4,0,inf,0,4,-inf,0,4,-inf
4,5,5,2.321928,0,inf,5,0,inf,2,3,-0.584963,4,1,2.000000,0,5,-inf,0,5,-inf
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4794,4795,3882,12.227315,913,2.088113,4350,445,3.289138,3304,1491,1.147933,1492,3303,-1.146529,914,3881,-2.086162,446,4349,-3.285568
4795,4796,3883,12.227616,913,2.088485,4351,445,3.289470,3305,1491,1.148370,1492,3304,-1.146966,914,3882,-2.086534,446,4350,-3.285900
4796,4797,3884,12.227917,913,2.088856,4352,445,3.289801,3306,1491,1.148806,1492,3305,-1.147403,914,3883,-2.086906,446,4351,-3.286231
4797,4798,3885,12.228217,913,2.089228,4353,445,3.290133,3306,1492,1.147839,1493,3305,-1.146436,914,3884,-2.087277,446,4352,-3.286563


In [5]:
# tabel, kus on verb+kääne koos graafiku infoga ja uniq lemma arvuga

query = f"SELECT * FROM verb_case_log_location"

verb_case_log_location = pd.read_sql(query, conn)
verb_case_log_location

,verb,verb_compound,morph_case,log2_tag,log2_annotation,verb_case_count,unique_lemmas,log2_unique_lemmas,synset_count
0,aasima,,ad,-9.965784,-2.807355,8,1,0.000000,2-3
1,aasima,,in,-1.000000,-0.415037,7,3,1.584963,2-3
2,abielluma,,abl,9.965784,-5.658211,103,2,1.000000,1
3,abielluma,,ad,-2.850307,-0.822241,1182,84,6.392317,1
4,abielluma,,adit,-3.321928,-0.125531,23,2,1.000000,1
...,...,...,...,...,...,...,...,...,...
21264,šokeerima,,ad,-3.544321,-0.921997,110,26,4.700440,2-3
21265,šokeerima,,all,9.965784,-2.321928,6,1,0.000000,2-3
21266,šokeerima,,el,-9.965784,-1.584963,16,4,2.000000,2-3
21267,šokeerima,,in,1.584963,-0.536053,49,16,4.000000,2-3


## merge

In [6]:
new_df = pd.merge(verb_case_log_location, lines_df, left_on='unique_lemmas', right_on='x')
new_df

,verb,verb_compound,morph_case,log2_tag,log2_annotation,verb_case_count,unique_lemmas,log2_unique_lemmas,synset_count,x,...,log2_y_pos70,y_pos30,y_neg30,log2_y_pos30,y_pos20,y_neg20,log2_y_pos20,y_pos10,y_neg10,log2_y_pos10
0,aasima,,ad,-9.965784,-2.807355,8,1,0.000000,2-3,1,...,-inf,1,0,inf,0,1,-inf,0,1,-inf
1,abistama,,abl,-9.965784,-2.584963,7,1,0.000000,2-3,1,...,-inf,1,0,inf,0,1,-inf,0,1,-inf
2,abistama,,all,-9.965784,-3.906891,16,1,0.000000,2-3,1,...,-inf,1,0,inf,0,1,-inf,0,1,-inf
3,adresseerima,,in,-9.965784,-2.584963,7,1,0.000000,1,1,...,-inf,1,0,inf,0,1,-inf,0,1,-inf
4,aeglustama,,all,-9.965784,-3.000000,9,1,0.000000,1,1,...,-inf,1,0,inf,0,1,-inf,0,1,-inf
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21264,õnnestuma,,in,1.227736,-0.736218,2058,312,8.285402,2-3,312,...,0.938013,108,204,-0.917538,51,261,-2.355481,23,289,-3.651364
21265,õppima,,in,4.905379,0.016435,11763,547,9.095397,2-3,547,...,1.003958,183,364,-0.992095,94,453,-2.268778,43,504,-3.551015
21266,ütlema,,ad,-5.088166,0.372719,22466,459,8.842350,2-3,459,...,0.985879,155,304,-0.971803,78,381,-2.288245,36,423,-3.554589
21267,ütlema,,el,0.974385,-0.987351,4271,629,9.296916,2-3,629,...,1.017237,209,420,-1.006886,109,520,-2.254183,51,578,-3.502500


## millesesse tsooni punkt kuulub

kas >= 90

<90 ja >=80

<= 70 ja >= 30

<20

<10


In [7]:
def get_level(row):
    if row['log2_tag'] >= row[f"log2_y_pos90"]: #kõrgemal 90 joonest
        return "n90"
    elif row['log2_tag'] >= row[f"log2_y_pos80"] and row['log2_tag'] < row[f"log2_y_pos90"]: #kõrgemal 80 joonest
        return "n80"
    elif row['log2_tag'] >= 0 and row['log2_tag'] <= row[f"log2_y_pos70"]: #madalamal 70 ja kõrgemal 0 joonest
        return "n70"
    elif row['log2_tag'] >= row[f"log2_y_pos30"] and row['log2_tag'] <= 0: #madalamal 0 ja kõrgemal 30 joonest
        return "n30"
    elif row['log2_tag'] <= row[f"log2_y_pos20"] and row['log2_tag'] > row[f"log2_y_pos10"]: #madalamal 20 joonest
        return "n20"
    elif row['log2_tag'] <= row[f"log2_y_pos10"]: #madalamal 10 joonest
        return "n10"
    else:
        return "-"

In [8]:
new_df["level"] = new_df.apply(get_level, axis=1)

## distance, ehk kui kõrgel on punkt joonest



def distance(row):
    if row["level"] == "n90":
        return row['log2_tag']-row[f"log2_y_pos90"]
    if row["level"] == "n80":
        return row['log2_tag']-row[f"log2_y_pos80"]
    if row["level"] == "n10":
        return row['log2_tag']-row[f"log2_y_pos10"]
    if row["level"] == "n20":
        return row['log2_tag']-row[f"log2_y_pos20"]
    if row["level"] == "n70":
        return row[f"log2_y_pos70"]- row['log2_tag']
    if row["level"] == "n30":
        return row[f"log2_y_pos30"]- row['log2_tag']

In [10]:
#new_df["distance"] = new_df.apply(distance, axis=1)

## not_ann_words ja ann_words 

(mitte lemmad vaid spatial_obl tabelist 'form' kus ekilex_tag on kas null või mitte)

In [9]:
query = """
        SELECT verb, verb_compound, morph_case, count(form) as not_ann_words
        FROM spatial_obl
        WHERE ekilex_tag is null
        GROUP BY verb, verb_compound, morph_case;
        """

df_notag = pd.read_sql(query, conn)
df_notag

,verb,verb_compound,morph_case,not_ann_words
0,0muutuma,,el,1
1,0olema,,ad,1
2,0olema,,el,2
3,0olema,,in,2
4,10halama,,in,1
...,...,...,...,...
65387,šveitsima,,in,6
65388,žestikuleerima,,ad,1
65389,žongleerima,,ad,3
65390,žongleerima,,in,2


In [10]:
# Merge with df_log based on verb, verb_compound, morph_case
new_df = new_df.merge(df_notag, on=['verb', 'verb_compound', 'morph_case'], how='left')

In [11]:
tags = "('location')"

query = f"""
        SELECT verb, verb_compound, morph_case, count(form) as ann_words
        FROM spatial_obl
        WHERE ekilex_tag IN {tags}
        GROUP BY verb, verb_compound, morph_case;
        """

df_tag = pd.read_sql(query, conn)
df_tag

,verb,verb_compound,morph_case,ann_words
0,21olema,,in,1
1,A. kollama,,ad,1
2,A. tihkama,,abl,1
3,B. kurtma,,in,1
4,B. teadma,,in,1
...,...,...,...,...
26158,šokeerima,,adit,1
26159,šokeerima,,all,1
26160,šokeerima,,ill,1
26161,šokeerima,,in,15


In [12]:
# Merge with df_log based on verb, verb_compound, morph_case
new_df = new_df.merge(df_tag, on=['verb', 'verb_compound', 'morph_case'], how='left')

## kui palju unique lemmasid ära katab

In [13]:
tags = "('location')"

query = f"""
    SELECT 
        verb, 
        verb_compound,
        morph_case, 
        COUNT(DISTINCT lemma) AS ann_unique_lemmas
    FROM spatial_obl
    WHERE ekilex_tag in {tags}
    GROUP BY verb, verb_compound, morph_case
"""

df_ul_tag = pd.read_sql(query, conn)
df_ul_tag

,verb,verb_compound,morph_case,ann_unique_lemmas
0,21olema,,in,1
1,A. kollama,,ad,1
2,A. tihkama,,abl,1
3,B. kurtma,,in,1
4,B. teadma,,in,1
...,...,...,...,...
26158,šokeerima,,adit,1
26159,šokeerima,,all,1
26160,šokeerima,,ill,1
26161,šokeerima,,in,11


In [14]:
# Merge with df_log based on verb, verb_compound, morph_case
new_df = new_df.merge(df_ul_tag, on=['verb', 'verb_compound', 'morph_case'], how='left')

## kui palju unique lemmasid on annoteerimata

In [15]:
query = f"""
    SELECT 
        verb, 
        verb_compound,
        morph_case, 
        COUNT(DISTINCT lemma) AS not_ann_unique_lemmas
    FROM spatial_obl
    WHERE ekilex_tag is null
    GROUP BY verb, verb_compound, morph_case
"""

df_ul_ntag = pd.read_sql(query, conn)
df_ul_ntag

,verb,verb_compound,morph_case,not_ann_unique_lemmas
0,0muutuma,,el,1
1,0olema,,ad,1
2,0olema,,el,2
3,0olema,,in,2
4,10halama,,in,1
...,...,...,...,...
65387,šveitsima,,in,6
65388,žestikuleerima,,ad,1
65389,žongleerima,,ad,2
65390,žongleerima,,in,2


In [16]:
new_df = new_df.merge(df_ul_ntag, on=['verb', 'verb_compound', 'morph_case'], how='left')

## olulisus (p-value)


"""
def get_min_success(n=100, p=0.8, kv=0.05):

    #n = 100       # number of trials
    #p = 0.8       # null hypothesis success rate

    # Find smallest k such that P(X ≥ k) < 0.05
    for k in range(n + 1):
        if binom.sf(k - 1, n, p) <= kv: #kv=0.05 95% puhul, 70% puhul peaks olema 0.05 asemel 0.95
            #print(f"Minimum k: {k}")
            #break
            return k
    
    #return np.nan
    return n
"""

def kv_for_datapoint(n, log2_ratio, p=0.8): #(log2_unique_lemmas->unique_lemmas, log2_tag, p)
    #n=2**log2_x
    # Compute observed successes from log2(y_pos/y_neg)
    ratio = 2 ** log2_ratio
    k_obs = n * ratio / (1 + ratio)
    # Compute probability P(X >= k_obs)
    kv_obs = binom.sf(int(round(k_obs)) - 1, int(round(n)), p)
    return n, k_obs, kv_obs

### Example: line point at (11.16, 2.13) corresponds to 5% line
### Above-point at (11.16, 3.06)
n_line, k_line, kv_line = kv_for_datapoint(1534, 2.163039, p=0.8)
n_obs, k_obs, kv_obs = kv_for_datapoint(1534, 2.236495, p=0.8) # (log2_unique_lemmas, log2_tag, p)

print(f"n (unique lemmas) = {n_line:.0f}")
print(f"Critical line point: k = {k_line:.0f} (y_pos80), kv = {kv_line:.4f}")
print(f"Above point: k = {k_obs:.0f}, kv = {kv_obs:.6f}")

**CDF = left side = P(X ≤ k)**

**SF = right side = P(X > k)**

**SF(k-1) = P(X ≥ k)**

**use CDF for lower tail, SF for upper tail**

In [17]:
mapping_level = {"n80": 0.8, "n90": 0.9, "n70":0.7, "n20": 0.2, "n10":0.1, "n30": 0.3}

def kv_for_datapoint(row): #(log2_unique_lemmas->unique_lemmas, log2_tag, p)
    #n=2**log2_x : log2_x = log2_unique_lemmas
    # n : unique_lemmas
    # log2_ratio : log2_tag
    # p = 0.8 kui n80
    
    if row["level"] != "-":
        n = row["unique_lemmas"]
        log2_ratio = row["log2_tag"]
        p = mapping_level[row["level"]]

        # Compute observed successes from log2(y_pos/y_neg)
        ratio = 2 ** log2_ratio
        k_obs = n * ratio / (1 + ratio)
        # Compute probability P(X >= k_obs), üleval pool joont
        if p>0.5:
            kv_obs = binom.sf(int(round(k_obs)) - 1, int(round(n)), p)
        else: # all pool joont
            kv_obs = binom.cdf(int(round(k_obs)), int(round(n)), p)
        #return n, k_obs, kv_obs
        return round(kv_obs, 5)
    else:
        return "-"

In [18]:
new_df["olulisus"] = new_df.apply(kv_for_datapoint, axis=1)

In [19]:
new_df = new_df.rename(columns={'log2_tag': 'ratio'})

In [20]:
new_df

,verb,verb_compound,morph_case,ratio,log2_annotation,verb_case_count,unique_lemmas,log2_unique_lemmas,synset_count,x,...,log2_y_pos20,y_pos10,y_neg10,log2_y_pos10,level,not_ann_words,ann_words,ann_unique_lemmas,not_ann_unique_lemmas,olulisus
0,aasima,,ad,-9.965784,-2.807355,8,1,0.000000,2-3,1,...,-inf,0,1,-inf,-,7.0,NaN,NaN,5.0,-
1,abistama,,abl,-9.965784,-2.584963,7,1,0.000000,2-3,1,...,-inf,0,1,-inf,-,6.0,NaN,NaN,4.0,-
2,abistama,,all,-9.965784,-3.906891,16,1,0.000000,2-3,1,...,-inf,0,1,-inf,-,15.0,NaN,NaN,12.0,-
3,adresseerima,,in,-9.965784,-2.584963,7,1,0.000000,1,1,...,-inf,0,1,-inf,-,6.0,NaN,NaN,3.0,-
4,aeglustama,,all,-9.965784,-3.000000,9,1,0.000000,1,1,...,-inf,0,1,-inf,-,8.0,NaN,NaN,7.0,-
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21264,õnnestuma,,in,1.227736,-0.736218,2058,312,8.285402,2-3,312,...,-2.355481,23,289,-3.651364,-,1286.0,541.0,209.0,529.0,-
21265,õppima,,in,4.905379,0.016435,11763,547,9.095397,2-3,547,...,-2.268778,43,504,-3.551015,n90,5848.0,5724.0,465.0,968.0,0.0
21266,ütlema,,ad,-5.088166,0.372719,22466,459,8.842350,2-3,459,...,-2.288245,36,423,-3.554589,n10,9790.0,362.0,112.0,1118.0,0.0
21267,ütlema,,el,0.974385,-0.987351,4271,629,9.296916,2-3,629,...,-2.254183,51,578,-3.502500,n70,2839.0,949.0,337.0,1192.0,0.97997


## salvestada andmebaasi

In [21]:
new_df.to_sql("lines_class_info3", conn, if_exists="replace", index=False)

21269

## uuesti sisse lugemine

In [22]:
# log2_unique_lemmas on sama, mis log2_x
# log2_tag on andmepunkti y 

query = """SELECT verb, verb_compound, morph_case, ratio, unique_lemmas, level, not_ann_words, ann_words, ann_unique_lemmas, not_ann_unique_lemmas, olulisus
            FROM lines_class_info3
            """

df = pd.read_sql(query, conn)

In [23]:
df

,verb,verb_compound,morph_case,ratio,unique_lemmas,level,not_ann_words,ann_words,ann_unique_lemmas,not_ann_unique_lemmas,olulisus
0,aasima,,ad,-9.965784,1,-,7.0,NaN,NaN,5.0,-
1,abistama,,abl,-9.965784,1,-,6.0,NaN,NaN,4.0,-
2,abistama,,all,-9.965784,1,-,15.0,NaN,NaN,12.0,-
3,adresseerima,,in,-9.965784,1,-,6.0,NaN,NaN,3.0,-
4,aeglustama,,all,-9.965784,1,-,8.0,NaN,NaN,7.0,-
...,...,...,...,...,...,...,...,...,...,...,...
21264,õnnestuma,,in,1.227736,312,-,1286.0,541.0,209.0,529.0,-
21265,õppima,,in,4.905379,547,n90,5848.0,5724.0,465.0,968.0,0.0
21266,ütlema,,ad,-5.088166,459,n10,9790.0,362.0,112.0,1118.0,0.0
21267,ütlema,,el,0.974385,629,n70,2839.0,949.0,337.0,1192.0,0.97997


In [24]:
df2 = df[df["level"]!= "-"]

In [25]:
df3 = df2[df2["level"]=="n80"]

In [26]:
df3 = df3.sort_values(["olulisus"])

In [27]:
df3

,verb,verb_compound,morph_case,ratio,unique_lemmas,level,not_ann_words,ann_words,ann_unique_lemmas,not_ann_unique_lemmas,olulisus
21249,valitsema,,in,3.044781,677,n80,3776.0,1865.0,551.0,1234.0,0.0
21223,toimuma,,in,2.873490,2200,n80,24937.0,22095.0,1871.0,4539.0,0.0
19354,treenima,,in,3.900242,142,n80,387.0,433.0,124.0,198.0,0.0
19486,õpetama,,in,3.626783,219,n80,788.0,630.0,181.0,317.0,0.0
19531,kasvama,üles,in,3.798366,172,n80,391.0,320.0,158.0,136.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...
18864,põgenema,,adit,4.285402,83,n80,231.0,351.0,72.0,90.0,8.0e-05
19931,elutsema,,in,4.554589,74,n80,150.0,94.0,70.0,105.0,8.0e-05
19940,müüma,,ill,4.610498,74,n80,84.0,171.0,67.0,53.0,8.0e-05
19943,pöörduma,,adit,4.643856,74,n80,416.0,325.0,63.0,152.0,8.0e-05


In [28]:
df3[df3["verb"]=="jääma"]

,verb,verb_compound,morph_case,ratio,unique_lemmas,level,not_ann_words,ann_words,ann_unique_lemmas,not_ann_unique_lemmas,olulisus
18428,jääma,maha,ill,6.321928,37,n80,33.0,80.0,36.0,30.0,0.00026
19599,jääma,edasi,ill,9.965784,25,n80,87.0,42.0,25.0,39.0,0.00378
20612,jääma,,ill,2.372398,695,n80,8500.0,2967.0,510.0,1280.0,0.0051
15344,jääma,alles,ill,4.930737,31,n80,113.0,61.0,29.0,88.0,0.00867
15345,jääma,maha,all,4.209453,31,n80,90.0,37.0,29.0,68.0,0.03745


In [29]:
df3[df3["verb"]=="viima"]

,verb,verb_compound,morph_case,ratio,unique_lemmas,level,not_ann_words,ann_words,ann_unique_lemmas,not_ann_unique_lemmas,olulisus
20977,viima,,ill,3.446670,710,n80,2316.0,3489.0,592.0,813.0,0.0
19691,viima,edasi,ill,9.965784,25,n80,19.0,33.0,25.0,17.0,0.00378
17979,viima,kaasa,el,5.209453,32,n80,66.0,74.0,30.0,41.0,0.00713


In [30]:
conn.close()